# Agreement for new independent ratings

Creates matched blank forms and evaluates only genuinely independent completed ratings.

Run the single code cell below with the **Python (skin)** kernel.

In [ ]:
from pathlib import Path
import sys

PACKAGE_DIR_INPUT = Path(r"D:\AIUB\Neural\A_Comprehensive_Framework_for_Auditing_Demographic_Bias_in_Synthetic_Skin_Disease_Datasets\revision_package")
if not PACKAGE_DIR_INPUT.is_dir():
    raise FileNotFoundError(f"revision_package not found: {PACKAGE_DIR_INPUT}")
if str(PACKAGE_DIR_INPUT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_DIR_INPUT))

from pathlib import Path
import pandas as pd

from notebook_config import DATA_ROOT, RUN_ROOT
from notebook_support import ensure_core_outputs, run_once

# Change to True only after two people completed the two files independently,
# blinded to previous proxy labels and model outputs.
CONFIRM_INDEPENDENT_RATERS = False
ANNOTATION_SAMPLE_N = 100

audit_metadata, _ = ensure_core_outputs(RUN_ROOT)
templates = RUN_ROOT / "annotation_templates"
templates.mkdir(parents=True, exist_ok=True)
rater_a_blank = templates / "rater_a_blank.csv"
rater_b_blank = templates / "rater_b_blank.csv"
rater_a_filled = templates / "rater_a_filled.csv"
rater_b_filled = templates / "rater_b_filled.csv"

if not rater_a_blank.exists() or not rater_b_blank.exists():
    data = pd.read_csv(audit_metadata, dtype={"sample_id": str})
    n = min(ANNOTATION_SAMPLE_N, len(data))
    if n < len(data):
        from sklearn.model_selection import StratifiedShuffleSplit
        splitter = StratifiedShuffleSplit(n_splits=1, test_size=n, random_state=20261004)
        _, selected = next(splitter.split(data, data["label"].astype(str)))
        subset = data.iloc[selected].copy()
    else:
        subset = data.copy()
    form = pd.DataFrame({
        "sample_id": subset["sample_id"].astype(str),
        "image_path": [str(DATA_ROOT / sample_id) for sample_id in subset["sample_id"].astype(str)],
        "appearance_proxy": "",
        "gender_proxy": "",
        "age_proxy": "",
    }).sort_values("sample_id")
    form.to_csv(rater_a_blank, index=False)
    form.to_csv(rater_b_blank, index=False)
    print(f"Created matched blank forms ({len(form)} images):\n{rater_a_blank}\n{rater_b_blank}")

if not rater_a_filled.exists() or not rater_b_filled.exists():
    print("\nGive separate blank files to two raters. They must use the same written codebook and work independently.")
    print(f"Save completed files as:\n{rater_a_filled}\n{rater_b_filled}")
elif not CONFIRM_INDEPENDENT_RATERS:
    print("Filled files exist, but agreement was not run. Set CONFIRM_INDEPENDENT_RATERS=True only if independence/blinding are genuine.")
else:
    agreement_dir = run_once(
        "07_annotation_agreement.py",
        [
            "--rater-a", str(rater_a_filled),
            "--rater-b", str(rater_b_filled),
            "--confirm-independent",
            "--columns", "appearance_proxy,gender_proxy,age_proxy",
        ],
        RUN_ROOT / "annotation_agreement",
        ["agreement_summary.csv", "interpretation.json"],
    )
    print(pd.read_csv(agreement_dir / "agreement_summary.csv").to_string(index=False))
    print("\nThis is a new subset audit, not retrospective reliability of the original annotations.")
